RUNTIME: A100 (High-RAM) recommended, T4 if unavailable     
INSTRUCTIONS:

1. Set runtime to A100: Runtime → Change runtime type → GPU → A100
2. Run all cells in order: Runtime → Run all
3. When prompted, authorize Google Drive access
4. Model checkpoints are saved to Google Drive via:
```
python   model.save_pretrained(SAVE_DIR + "checkpoints/distilbert_model")
```
Make sure SAVE_DIR is set correctly in the setup cell   

5. To save results/figures to GitHub:
    a. Right-click the file in the left sidebar → Download  
    b. Go to the GitHub repo → results/ → Add file → Upload files   
    c. Commit directly on GitHub    
6. Save the notebook: Ctrl+S → save to GitHub when prompted     

⚠️ Colab sessions disconnect after periods of inactivity. Save checkpoints to Drive frequently during long training runs.

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────
!pip install -q -U "transformers<5" accelerate datasets

# ── 2. Optional Google Drive mount ────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = "/content/drive/MyDrive/cs4120_project/"
except Exception:
    SAVE_DIR = None

# ── 3. Standard imports ───────────────────────────────────────────
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.metrics import f1_score, classification_report
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ── 4. Resolve repo/data/results paths ────────────────────────────
repo_candidates = [Path.cwd(), Path.cwd().parent]
repo_root = None
for candidate in repo_candidates:
    if (candidate / "src").exists() and (candidate / "data").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not locate repo root with src/ and data/ directories")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

DATA_DIR = repo_root / "data"
RESULTS_DIR = repo_root / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── 5. Sanity checks ──────────────────────────────────────────────
print(f"PyTorch: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Repo root:", repo_root)
print("Data dir:", DATA_DIR)
print("Results dir:", RESULTS_DIR)

Mounted at /content/drive
PyTorch: 2.10.0+cu128
GPU available: True
GPU: NVIDIA A100-SXM4-40GB


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import ast
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MultiLabelBinarizer
import ast, json, time
from pathlib import Path
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import MultiLabelBinarizer
from torch.utils.data import Dataset
from transformers import (
   AutoTokenizer,
   AutoModelForSequenceClassification,
   TrainingArguments,
   Trainer,
   EarlyStoppingCallback,
   set_seed,
)


from pathlib import Path
data_directory = Path("/content/drive/MyDrive/cs4120_project/data")
results_directory = Path("/content/drive/MyDrive/cs4120_project/results")
results_directory.mkdir(parents=True, exist_ok=True)
train_df = pd.read_csv(data_directory / "train.csv")
val_df = pd.read_csv(data_directory / "val.csv")
test_df = pd.read_csv(data_directory / "test.csv")
fractions = [0.01, 0.05, 0.10, 0.25, 0.50, 1.0]
epochs = [1, 3, 5]
learning_rates = [1e-5, 2e-5, 5e-5]
batch_sizes = [16, 32, 64]
weight_decays = [0.01, 0.1, 0.001]
seeds = [42, 7, 21]




#Converts string to list
def parse_labels(x):
   if isinstance(x, list):
       return x
   try:
       return ast.literal_eval(x)
   except:
       return []


for df in [train_df, val_df, test_df]:
   df["labels"] = df["labels"].apply(parse_labels)


text_col = "text_clean_transformer"
for df in [train_df, val_df, test_df]:
   df.rename(columns={text_col: "text"}, inplace=True)


all_labels = set()
for labels in train_df["labels"]:
   all_labels.update(labels)


label_classes = sorted(list(all_labels))  
num_labels = len(label_classes)
label_names = label_classes



mlb = MultiLabelBinarizer(classes=label_classes)
mlb.fit([label_classes])


for df in [train_df, val_df, test_df]:
   df["labels_binary"] = list(mlb.transform(df["labels"]))



In [ ]:
#Dataset Class
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")


class EmotionDataset(Dataset):
   def __init__(self, df):
       self.encodings = tokenizer(
           df["text"].tolist(),
           truncation=True,
           padding="max_length",
           max_length=128,
           return_tensors="pt"
       )
       self.labels = torch.tensor(np.stack(df["labels_binary"]), dtype=torch.float32)


   def __len__(self):
       return len(self.labels)


   def __getitem__(self, idx):
       return {
           "input_ids":      self.encodings["input_ids"][idx],
           "attention_mask": self.encodings["attention_mask"][idx],
           "labels":         self.labels[idx],
       }



#get f1s
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    
    probs = torch.sigmoid(torch.tensor(logits)).numpy()

    preds = (probs >= 0.5).astype(int)
    
    f1_micro = f1_score(labels.astype(int), preds, average="micro", zero_division=0)
    f1_macro = f1_score(labels.astype(int), preds, average="macro", zero_division=0)
    
    return {"f1_micro": f1_micro, "f1_macro": f1_macro}


#Function to fun the expiernemnts
def run(train_df, val_df, test_df, lr, bs, ep, wd, seed, label):
   set_seed(seed)
   t0 = time.time()
  
   train_ds = EmotionDataset(train_df)
   val_ds = EmotionDataset(val_df)
   test_ds = EmotionDataset(test_df)
  
   model = AutoModelForSequenceClassification.from_pretrained(
       "distilbert-base-uncased", num_labels=num_labels, problem_type="multi_label_classification"
   )
  
   args = TrainingArguments(
       output_dir=f"/tmp/{label}",
       num_train_epochs=ep,
       per_device_train_batch_size=bs,
       per_device_eval_batch_size=bs*2,
       learning_rate=lr,
       weight_decay=wd,
       evaluation_strategy="epoch",
       save_strategy="no",
       logging_strategy="no",
       seed=seed,
       fp16=torch.cuda.is_available(),
   )
  
   trainer = Trainer(
       model=model,
       args=args,
       train_dataset=train_ds,
       eval_dataset=val_ds,
       compute_metrics=compute_metrics,
   )
  
   trainer.train()
  
   test_pred = trainer.predict(test_ds)
   logits_np = test_pred.predictions
   labels_np = test_pred.label_ids
   preds_np = (torch.sigmoid(torch.tensor(logits_np)) >= 0.5).int().numpy()
  
   f1_micro = f1_score(labels_np, preds_np, average="micro", zero_division=0)
   f1_macro = f1_score(labels_np, preds_np, average="macro", zero_division=0)
   elapsed = (time.time() - t0) / 60
  
   if label == "frac=100pct_seed42":
       if SAVE_DIR:
            model.save_pretrained(f"{SAVE_DIR}checkpoints/distilbert_model")
            tokenizer.save_pretrained(f"{SAVE_DIR}checkpoints/distilbert_model")
  
   return {
       "label": label,
       "n_train": len(train_df),
       "lr": lr,
       "batch_size": bs,
       "epochs": ep,
       "weight_decay": wd,
       "test_f1_micro": f1_micro,
       "test_f1_macro": f1_macro,
       "elapsed_min": elapsed,
       "y_pred": preds_np,
       "y_true": labels_np,
   }


In [ ]:
hp_results = []

#test out every hyperaprermter
for lr in learning_rates:
   r = run(train_df, val_df, test_df, lr=lr, bs=32, ep=3, wd=0.01, seed=42, label=f"lr_{lr:.0e}")
   r["param_type"] = "learning_rate"
   hp_results.append(r)


for bs in batch_sizes:
   r = run(train_df, val_df, test_df, lr=2e-5, bs=bs, ep=3, wd=0.01, seed=42, label=f"bs_{bs}")
   r["param_type"] = "batch_size"
   hp_results.append(r)


for ep in epochs:
   r = run(train_df, val_df, test_df, lr=2e-5, bs=32, ep=ep, wd=0.01, seed=42, label=f"ep_{ep}")
   r["param_type"] = "epochs"
   hp_results.append(r)


for wd in weight_decays:
   r = run(train_df, val_df, test_df, lr=2e-5, bs=32, ep=3, wd=wd, seed=42, label=f"wd_{wd}")
   r["param_type"] = "weight_decay"
   hp_results.append(r)


hp_df = pd.DataFrame(hp_results)
#get best hyperparams
best_lr = hp_df[hp_df["param_type"] == "learning_rate"].nlargest(1, "test_f1_micro")["lr"].values[0]
best_bs = int(hp_df[hp_df["param_type"] == "batch_size"].nlargest(1, "test_f1_micro")["batch_size"].values[0])
best_ep = int(hp_df[hp_df["param_type"] == "epochs"].nlargest(1, "test_f1_micro")["epochs"].values[0])
best_wd = hp_df[hp_df["param_type"] == "weight_decay"].nlargest(1, "test_f1_micro")["weight_decay"].values[0]




hp_df.insert(0, "method", "distilbert")
hp_df.to_csv(RESULTS_DIR / "distilbert_hparam.csv", index=False)

In [ ]:
frac_results = []


for frac in fractions:
   for seed in seeds:
       df_frac = train_df.sample(frac=frac, random_state=seed).reset_index(drop=True)
       r = run(
           df_frac, val_df, test_df,
           lr=best_lr, bs=best_bs, ep=best_ep, wd=best_wd,
           seed=seed,
           label=f"frac={int(frac*100)}pct_seed{seed}"
       )
       r["frac"] = frac
       r["seed"] = seed
       frac_results.append(r)


frac_df = pd.DataFrame(frac_results)
print(frac_df[["frac", "seed", "test_f1_micro", "test_f1_macro"]])
frac_df.insert(0, "method", "distilbert")
frac_df.to_csv(RESULTS_DIR / "distilbert_fractions.csv", index=False)


In [ ]:
from src.evaluate import evaluate_run



best_run = frac_df[frac_df["frac"] == 1.0].iloc[0]
y_test = best_run["y_true"]
y_pred_best = best_run["y_pred"]


ev = evaluate_run(
   method="distilbert",
   data_fraction=1.0,
   seed=42,
   label_names=label_names,
   y_true=y_test,
   y_pred=y_pred_best,
)


ev["per_class"].to_csv(RESULTS_DIR / "distilbert_per_class.csv", index=False)
print(ev["per_class"][["emotion", "f1", "precision", "recall"]].sort_values("f1", ascending=False))
